In [3]:
"""
Stage 4 Assembly -- 07: Drop Binary Regime Indicators
=======================================================
Removes the five binary regime indicators from every assembled table and
every split, in place, after the panel macro-feature fix (notebooks 03-05
already re-run). No re-normalisation, no re-assembly.

THE FIVE FEATURES
------------------
    vix_above_20            = 1{vix > 20}
    vix_above_30             = 1{vix > 30}
    curve_inverted_2y10y     = 1{slope_2y10y < 0}
    curve_inverted_3m10y     = 1{slope_3m10y < 0}
    credit_stress            = 1{hy_oas > 5}

Each is a hard threshold on a feature that is ALREADY present, continuous,
and z-scored elsewhere in the same dataset (vix, slope_2y10y, slope_3m10y,
hy_oas respectively). A B-spline basis over the continuous parent can
represent any threshold effect the binary version could express, and can
also represent thresholds the binary version cannot (a different cutoff,
a smooth transition, an asymmetric response either side of the cutoff).
The binary versions therefore add no information the model cannot already
recover from features it already has.

They were also structurally different from every other feature in the
pipeline in a way that made them awkward passengers rather than first-class
citizens: never z-scored (SKIP_ZSCORE in 04_normalise), exempted from every
R0-R6 exclusion rule (exempt=SKIP_ZSCORE in apply_rules), and excluded from
the clip in finalise(). None of that data treatment changes for any other
feature by removing them now -- every other feature's warm-up, sigma_f,
Welford state and exclusion-rule outcome is computed independently per
feature, so this is a pure column deletion with no side effects elsewhere.

WHY IN-PLACE, NOT A RE-RUN
---------------------------
Re-running Stage 3 normalisation and Stage 4 assembly would reproduce
byte-identical output for every other column -- there is nothing for the
five binaries' removal to disturb upstream or downstream, since they were
never inputs to any other feature's computation. A full re-run to delete
five known columns would cost hours for zero additional correctness.

The corresponding CODE change (so a future from-scratch run reproduces this
without needing this notebook) is:
    lib/config.py           -- BINARIES = []  (old list kept as LEGACY_BINARIES)
    01_apply_union.ipynb    -- add the five names to MANUAL_DROP

Both are one-line-ish edits, described at the end of this notebook, and are
NOT applied automatically here -- they are a separate, deliberate step you
make once you're satisfied this notebook's output is correct.

SCOPE
-----
02_assembled/   agg_means, agg_full_moments, panel  (+ their *_nan siblings)  -- 6 files
04_splits/      Split_{A,B,C,D} x {agg_means, agg_full_moments, panel} x
                {train, val, test}                                           -- 36 files
fill reports    fill_report_aggregate.csv, fill_report_panel.csv (drop rows)

01_unioned/ is deliberately left untouched. Nothing downstream reads it, and
hand-editing it would make it inconsistent with what its own notebook (01)
produces on a fresh run -- exactly the kind of drift this notebook exists
to avoid introducing elsewhere.

All five binaries are macro raw_level features and therefore appear, under
their bare names, in EVERY dataset that carries macro features -- agg_means,
agg_full_moments, AND panel all include them. (An earlier version of this
notebook assumed agg_full_moments carried none, on the theory that macro
features are always suffixed there like stock moments are; that assumption
was wrong -- macro raw_level features are never suffixed anywhere, so they
appear under the same bare name in every table.)

IDEMPOTENCY
-----------
drop_binaries() tolerates a file already being clean (0 binaries present),
since a first run of this notebook may be interrupted partway through Step 1
or Step 2 -- exactly what happened during initial development, when the
notebook crashed after processing agg_means/agg_means_nan but before
reaching agg_full_moments. Re-running afterwards must not treat "already
correctly dropped" as an error. Any count other than {expected_n, 0} is
still a hard failure, since that is a genuinely inconsistent state.
"""

import sys
from pathlib import Path

import pandas as pd

sys.path.append('../..')
from lib.config import OUT

ASM_DIR = OUT / '02_assembled'
SPLIT_DIR = OUT / '04_splits'

BINARIES = ['vix_above_20', 'vix_above_30', 'curve_inverted_2y10y',
            'curve_inverted_3m10y', 'credit_stress']

SPLITS = ['Split_A', 'Split_B', 'Split_C', 'Split_D']
PARTS = ['train', 'val', 'test']

# All three datasets carry all five binaries under their bare macro names.
EXPECTED_N_BINARIES = {
    'agg_means': 5,
    'agg_full_moments': 5,
    'panel': 5,
}


def drop_binaries(path: Path, expected_n: int) -> tuple[int, int]:
    """
    Load a parquet, drop any of the five binaries present, overwrite in place.

    Tolerates two valid states: expected_n present (fresh file), or 0 present
    (already dropped by a prior partial run of this notebook). Any OTHER
    count is a genuine inconsistency and still raises.

    Returns (n_before, n_dropped). n_dropped is 0 if the file was already
    clean -- this is not itself a failure, just nothing left to do.
    """
    df = pd.read_parquet(path)
    n_before = df.shape[1]

    present = [c for c in BINARIES if c in df.columns]
    assert len(present) in (expected_n, 0), (
        f"{path.name}: expected {expected_n} binaries present (or 0 if "
        f"already dropped by a prior run), found {len(present)} "
        f"({present}). This is an inconsistent state -- investigate before "
        f"re-running.")

    if not present:
        return n_before, 0

    df = df.drop(columns=present)
    df.to_parquet(path, index=False)
    return n_before, len(present)


# ═══════════════════════════════════════════════════════════════════════════════
# STEP 1: DROP FROM 02_assembled
# ═══════════════════════════════════════════════════════════════════════════════

print("=" * 100)
print("STAGE 4 ASSEMBLY -- 07: DROP BINARY REGIME INDICATORS")
print("=" * 100)
print(f"\n  Dropping: {BINARIES}")

print(f"\n{'-' * 100}")
print("STEP 1: 02_assembled/")
print(f"{'-' * 100}")

assembled_files = {
    'agg_means': ['agg_means.parquet', 'agg_means_nan.parquet'],
    'agg_full_moments': ['agg_full_moments.parquet', 'agg_full_moments_nan.parquet'],
    'panel': ['panel.parquet', 'panel_nan.parquet'],
}

total_dropped_assembled = 0
for dataset, fnames in assembled_files.items():
    expected = EXPECTED_N_BINARIES[dataset]
    for fname in fnames:
        path = ASM_DIR / fname
        n_before, n_dropped = drop_binaries(path, expected)
        total_dropped_assembled += n_dropped
        status = f"(dropped {n_dropped})" if n_dropped else "(already clean)"
        print(f"  {fname:<32} {n_before:>5} -> {n_before - n_dropped:>5} cols   "
              f"{status}")

print(f"\n  Total columns dropped this run: {total_dropped_assembled}")
print(f"  (30 would mean a fully fresh run; fewer means some files were "
      f"already clean from a prior partial run -- both are fine, as long "
      f"as Step 4 below confirms every file ends up clean)")


# ═══════════════════════════════════════════════════════════════════════════════
# STEP 2: DROP FROM 04_splits
# ═══════════════════════════════════════════════════════════════════════════════

print(f"\n{'-' * 100}")
print("STEP 2: 04_splits/")
print(f"{'-' * 100}")

total_dropped_splits = 0
n_files_touched = 0

for split in SPLITS:
    for dataset, expected in EXPECTED_N_BINARIES.items():
        for part in PARTS:
            fname = f'{dataset}_{part}.parquet'
            path = SPLIT_DIR / split / fname
            n_before, n_dropped = drop_binaries(path, expected)
            total_dropped_splits += n_dropped
            n_files_touched += 1

    print(f"  {split}: processed ({len(EXPECTED_N_BINARIES) * len(PARTS)} files)")

print(f"\n  Files processed: {n_files_touched}   (expected "
      f"{len(SPLITS) * len(EXPECTED_N_BINARIES) * len(PARTS)})")
assert n_files_touched == len(SPLITS) * len(EXPECTED_N_BINARIES) * len(PARTS)
print(f"  Total columns dropped this run: {total_dropped_splits}")
print(f"  (180 would mean a fully fresh run; fewer means some files were "
      f"already clean -- Step 4 confirms the final result regardless)")


# ═══════════════════════════════════════════════════════════════════════════════
# STEP 3: UPDATE FILL REPORTS
# ═══════════════════════════════════════════════════════════════════════════════
# fill_report_aggregate.csv / fill_report_panel.csv carry one row per feature
# per dataset (n_nan, pct_nan, n_clipped, ...). Rows for the five dropped
# features are removed so the reports describe exactly the features that
# still exist. Idempotent by construction -- filtering out rows that were
# already absent from a prior run is a no-op, not an error.

print(f"\n{'-' * 100}")
print("STEP 3: FILL REPORTS")
print(f"{'-' * 100}")

for report_name in ['fill_report_aggregate.csv', 'fill_report_panel.csv']:
    path = ASM_DIR / report_name
    if not path.exists():
        print(f"  {report_name}: not found, skipping")
        continue

    rep = pd.read_csv(path)
    n_before = len(rep)
    n_matching = rep['feature'].isin(BINARIES).sum()

    rep = rep[~rep['feature'].isin(BINARIES)].reset_index(drop=True)
    rep.to_csv(path, index=False)

    print(f"  {report_name:<32} {n_before:>5} -> {len(rep):>5} rows   "
          f"(dropped {n_matching})")


# ═══════════════════════════════════════════════════════════════════════════════
# STEP 4: FINAL VERIFICATION
# ═══════════════════════════════════════════════════════════════════════════════
# Re-read every touched file from disk (not from any in-memory state) and
# confirm zero binaries remain anywhere. This is the check that actually
# matters -- everything above only reports what happened THIS run, this
# step asserts the FINAL state is clean regardless of how many runs it took.

print(f"\n{'-' * 100}")
print("STEP 4: FINAL VERIFICATION (re-read from disk)")
print(f"{'-' * 100}")

all_clean = True

for dataset, fnames in assembled_files.items():
    for fname in fnames:
        cols = pd.read_parquet(ASM_DIR / fname).columns
        leftover = [c for c in BINARIES if c in cols]
        if leftover:
            all_clean = False
            print(f"  ✗ {fname}: still contains {leftover}")

for split in SPLITS:
    for dataset in EXPECTED_N_BINARIES:
        for part in PARTS:
            fname = f'{dataset}_{part}.parquet'
            cols = pd.read_parquet(SPLIT_DIR / split / fname).columns
            leftover = [c for c in BINARIES if c in cols]
            if leftover:
                all_clean = False
                print(f"  ✗ {split}/{fname}: still contains {leftover}")

if all_clean:
    print(f"\n  ✓ Confirmed: zero binary regime indicators remain in any "
          f"assembled table or split")
else:
    raise AssertionError(
        "One or more files still contain a binary regime indicator after "
        "the drop -- see ✗ lines above.")

print(f"\n  01_unioned/ was NOT touched (nothing downstream reads it).")


# ═══════════════════════════════════════════════════════════════════════════════
# SUMMARY
# ═══════════════════════════════════════════════════════════════════════════════

print(f"\n{'=' * 100}")
print("DONE")
print(f"{'=' * 100}")
print(f"""
  Dropped from: 02_assembled/ (6 files), 04_splits/ (36 files),
                fill_report_aggregate.csv, fill_report_panel.csv

  NOT yet done -- separate, deliberate step:

    lib/config.py:
        BINARIES = []
        LEGACY_BINARIES = ['vix_above_20', 'vix_above_30',
                            'curve_inverted_2y10y', 'curve_inverted_3m10y',
                            'credit_stress']

    Stage_4_Assembly/01_apply_union.ipynb, MANUAL_DROP:
        MANUAL_DROP |= {{
            'vix_above_20', 'vix_above_30',
            'curve_inverted_2y10y', 'curve_inverted_3m10y', 'credit_stress',
        }}
        # Each is a hard threshold on a feature already present and
        # continuous (vix, slope_2y10y, slope_3m10y, hy_oas respectively);
        # a spline over the parent subsumes any threshold effect the
        # binary could express. See Stage_4_Assembly/07_drop_binaries.ipynb
        # for the full rationale.

  These two edits make a from-scratch pipeline run reproduce today's result
  without needing this notebook again. They are not applied by this notebook
  on purpose -- confirm the drop above is correct first.
""")

STAGE 4 ASSEMBLY -- 07: DROP BINARY REGIME INDICATORS

  Dropping: ['vix_above_20', 'vix_above_30', 'curve_inverted_2y10y', 'curve_inverted_3m10y', 'credit_stress']

----------------------------------------------------------------------------------------------------
STEP 1: 02_assembled/
----------------------------------------------------------------------------------------------------
  agg_means.parquet                  576 ->   576 cols   (already clean)
  agg_means_nan.parquet              576 ->   576 cols   (already clean)
  agg_full_moments.parquet          1706 ->  1701 cols   (dropped 5)
  agg_full_moments_nan.parquet      1706 ->  1701 cols   (dropped 5)
  panel.parquet                      583 ->   578 cols   (dropped 5)
  panel_nan.parquet                  583 ->   578 cols   (dropped 5)

  Total columns dropped this run: 20
  (30 would mean a fully fresh run; fewer means some files were already clean from a prior partial run -- both are fine, as long as Step 4 below confi

In [4]:
rep = pd.read_csv(ASM_DIR / 'fill_report_aggregate.csv')
print(rep['feature'].isin(BINARIES).sum())   # should be 0 if genuinely never listed
print([c for c in BINARIES if c in rep['feature'].values])

0
[]
